# exemple de data augmentation sur une même image

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,
    img_to_array,
    load_img,
)


# -------------------------------------------------------------------------
# 1. Sélection de quatre radiographies différentes
# -------------------------------------------------------------------------
image_paths = [
    Path(
        "../../../COVID-19_Radiography_Dataset_CLAHE/"
        "COVID/COVID-1.png"
    ),
    Path(
        "../../../COVID-19_Radiography_Dataset_CLAHE/"
        "COVID/COVID-2.png"
    ),
    Path(
        "../../../COVID-19_Radiography_Dataset_CLAHE/"
        "COVID/COVID-3.png"
    ),
    Path(
        "../../../COVID-19_Radiography_Dataset_CLAHE/"
        "COVID/COVID-4.png"
    ),
]

for image_path in image_paths:
    if not image_path.exists():
        raise FileNotFoundError(
            f"Image introuvable : {image_path.resolve()}"
        )


# -------------------------------------------------------------------------
# 2. Dossier de sortie
# -------------------------------------------------------------------------
output_dir = Path("data_augmentation_animation")
output_dir.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------------------------------
# 3. Paramètres identiques à ceux du pipeline
# -------------------------------------------------------------------------
data_generator = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=5,
    horizontal_flip=True,
    fill_mode="nearest",
)


# -------------------------------------------------------------------------
# 4. Fonction de création de la légende
# -------------------------------------------------------------------------
def creer_legende_transformation(
    params: dict,
    image_width: int,
    image_height: int,
) -> str:
    """
    Transforme les paramètres internes de Keras en une légende lisible.

    Dans ImageDataGenerator :
    - theta : rotation en degrés ;
    - tx : translation verticale en pixels ;
    - ty : translation horizontale en pixels ;
    - shear : cisaillement en degrés ;
    - zx et zy : facteurs de zoom.
    """

    transformations = []

    theta = float(params.get("theta", 0) or 0)
    tx = float(params.get("tx", 0) or 0)
    ty = float(params.get("ty", 0) or 0)
    shear = float(params.get("shear", 0) or 0)
    zx = float(params.get("zx", 1) or 1)
    zy = float(params.get("zy", 1) or 1)

    flip_horizontal = bool(
        params.get("flip_horizontal", False)
    )
    flip_vertical = bool(
        params.get("flip_vertical", False)
    )

    if abs(theta) >= 0.05:
        transformations.append(
            f"rotation {theta:+.1f}°"
        )

    if abs(tx) >= 0.05:
        tx_percent = 100 * tx / image_height
        transformations.append(
            "translation verticale "
            f"{tx:+.1f} px ({tx_percent:+.1f} %)"
        )

    if abs(ty) >= 0.05:
        ty_percent = 100 * ty / image_width
        transformations.append(
            "translation horizontale "
            f"{ty:+.1f} px ({ty_percent:+.1f} %)"
        )

    zoom_x_percent = (zx - 1) * 100
    zoom_y_percent = (zy - 1) * 100

    if (
        abs(zoom_x_percent) >= 0.05
        or abs(zoom_y_percent) >= 0.05
    ):
        transformations.append(
            f"zoom x {zoom_x_percent:+.1f} %, "
            f"zoom y {zoom_y_percent:+.1f} %"
        )

    if abs(shear) >= 0.05:
        transformations.append(
            f"cisaillement {shear:+.1f}°"
        )

    if flip_horizontal:
        transformations.append("flip horizontal")

    if flip_vertical:
        transformations.append("flip vertical")

    if not transformations:
        return "Aucune transformation notable"

    return " ; ".join(transformations)


# -------------------------------------------------------------------------
# 5. Génération reproductible
# -------------------------------------------------------------------------
# Le générateur aléatoire produit un seed différent pour chaque image,
# tout en permettant de retrouver les mêmes résultats à chaque exécution.
rng = np.random.default_rng(42)

resultats = []


# -------------------------------------------------------------------------
# 6. Pour chaque radiographie :
#    - enregistrer l'originale ;
#    - générer UNE seule variante ;
#    - enregistrer les paramètres exacts.
# -------------------------------------------------------------------------
for index, image_path in enumerate(image_paths, start=1):

    image = load_img(
        image_path,
        color_mode="grayscale",
        target_size=(299, 299),
    )

    image_array = img_to_array(image).astype(np.float32)

    image_height, image_width = image_array.shape[:2]

    # -------------------------------------------------------------
    # Enregistrement de l'image originale
    # -------------------------------------------------------------
    original_filename = (
        f"image_{index:02d}_originale.png"
    )
    original_output_path = (
        output_dir / original_filename
    )

    plt.imsave(
        original_output_path,
        image_array.squeeze(),
        cmap="gray",
        vmin=0,
        vmax=255,
    )

    resultats.append(
        {
            "ordre": 2 * index - 1,
            "groupe": index,
            "type": "originale",
            "image_source": image_path.name,
            "fichier": original_filename,
            "theta_deg": 0.0,
            "translation_verticale_px": 0.0,
            "translation_horizontale_px": 0.0,
            "translation_verticale_pct": 0.0,
            "translation_horizontale_pct": 0.0,
            "shear_deg": 0.0,
            "zoom_x": 1.0,
            "zoom_y": 1.0,
            "flip_horizontal": False,
            "flip_vertical": False,
            "legende": (
                f"Radiographie {index} — image originale"
            ),
        }
    )

    # -------------------------------------------------------------
    # Génération d'une seule transformation aléatoire
    # -------------------------------------------------------------
    transformation_seed = int(
        rng.integers(0, 2**31 - 1)
    )

    params = data_generator.get_random_transform(
        img_shape=image_array.shape,
        seed=transformation_seed,
    )

    augmented_image = data_generator.apply_transform(
        image_array.copy(),
        params,
    )

    augmented_filename = (
        f"image_{index:02d}_augmentee.png"
    )
    augmented_output_path = (
        output_dir / augmented_filename
    )

    plt.imsave(
        augmented_output_path,
        augmented_image.squeeze(),
        cmap="gray",
        vmin=0,
        vmax=255,
    )

    theta = float(params.get("theta", 0) or 0)
    tx = float(params.get("tx", 0) or 0)
    ty = float(params.get("ty", 0) or 0)
    shear = float(params.get("shear", 0) or 0)
    zx = float(params.get("zx", 1) or 1)
    zy = float(params.get("zy", 1) or 1)

    details_transformation = (
        creer_legende_transformation(
            params=params,
            image_width=image_width,
            image_height=image_height,
        )
    )

    resultats.append(
        {
            "ordre": 2 * index,
            "groupe": index,
            "type": "augmentee",
            "image_source": image_path.name,
            "fichier": augmented_filename,
            "theta_deg": round(theta, 3),
            "translation_verticale_px": round(
                tx,
                3,
            ),
            "translation_horizontale_px": round(
                ty,
                3,
            ),
            "translation_verticale_pct": round(
                100 * tx / image_height,
                3,
            ),
            "translation_horizontale_pct": round(
                100 * ty / image_width,
                3,
            ),
            "shear_deg": round(shear, 3),
            "zoom_x": round(zx, 4),
            "zoom_y": round(zy, 4),
            "flip_horizontal": bool(
                params.get(
                    "flip_horizontal",
                    False,
                )
            ),
            "flip_vertical": bool(
                params.get(
                    "flip_vertical",
                    False,
                )
            ),
            "legende": (
                f"Radiographie {index} — "
                f"{details_transformation}"
            ),
        }
    )

    print(f"\nRadiographie {index}")
    print(f"Source : {image_path.name}")
    print(f"Transformation : {details_transformation}")


# -------------------------------------------------------------------------
# 7. Création du CSV utilisé par Streamlit
# -------------------------------------------------------------------------
df_transformations = (
    pd.DataFrame(resultats)
    .sort_values("ordre")
    .reset_index(drop=True)
)

csv_path = (
    output_dir
    / "transformations_data_augmentation.csv"
)

df_transformations.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
)


# -------------------------------------------------------------------------
# 8. Contrôle du résultat
# -------------------------------------------------------------------------
print("\nGénération terminée.")
print(
    f"Images enregistrées dans : "
    f"{output_dir.resolve()}"
)
print(
    f"Paramètres enregistrés dans : "
    f"{csv_path.resolve()}"
)

display(
    df_transformations[
        [
            "groupe",
            "type",
            "fichier",
            "legende",
        ]
    ]
)



Radiographie 1
Source : COVID-1.png
Transformation : rotation +5.9° ; translation verticale -1.6 px (-0.5 %) ; translation horizontale +1.3 px (+0.4 %) ; zoom x +2.1 %, zoom y -0.4 % ; cisaillement +4.9°

Radiographie 2
Source : COVID-2.png
Transformation : rotation +1.2° ; translation verticale +13.9 px (+4.7 %) ; translation horizontale -2.7 px (-0.9 %) ; zoom x -8.5 %, zoom y +5.5 % ; cisaillement -2.3°

Radiographie 3
Source : COVID-3.png
Transformation : rotation -6.0° ; translation verticale -4.3 px (-1.4 %) ; translation horizontale +10.3 px (+3.4 %) ; zoom x +7.7 %, zoom y +5.1 % ; cisaillement +1.3° ; flip horizontal

Radiographie 4
Source : COVID-4.png
Transformation : rotation +0.4° ; translation verticale -6.1 px (-2.0 %) ; translation horizontale +3.4 px (+1.1 %) ; zoom x -5.5 %, zoom y +3.0 % ; cisaillement +3.3° ; flip horizontal

Génération terminée.
Images enregistrées dans : C:\Users\n_a_e\Documents\DataScientest\Data Scientist\Projet COVID\Liora_Covid\notebooks\Prep

,groupe,type,fichier,legende
0,1,originale,image_01_originale.png,Radiographie 1 — image originale
1,1,augmentee,image_01_augmentee.png,Radiographie 1 — rotation +5.9° ; translation ...
2,2,originale,image_02_originale.png,Radiographie 2 — image originale
3,2,augmentee,image_02_augmentee.png,Radiographie 2 — rotation +1.2° ; translation ...
4,3,originale,image_03_originale.png,Radiographie 3 — image originale
5,3,augmentee,image_03_augmentee.png,Radiographie 3 — rotation -6.0° ; translation ...
6,4,originale,image_04_originale.png,Radiographie 4 — image originale
7,4,augmentee,image_04_augmentee.png,Radiographie 4 — rotation +0.4° ; translation ...
